In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

data = pd.read_csv('Twinsprings_alltime.csv')

print(data.head())

/var/folders/9n/npt_hczs5j115291dbmz7jnh0000gn/T/ipykernel_66352/1825050890.py:5: DtypeWarning: Columns (0: method_category, 1: qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('Twinsprings_alltime.csv')


                                     id approval_status  \
0  65073363-2d0f-483a-8783-3e3c2205e93d        Approved   
1  12a871d8-d5d0-47db-9bb0-95ebaa85d9d9        Approved   
2  347b8fca-7d81-45bf-9213-1b16033b2e6b        Approved   
3  25aaa074-ad6e-4780-a34e-0dcb4d1488e0        Approved   
4  a11ab19e-dff6-4883-ac96-291b6f08926a        Approved   

                      last_modified method_category monitoring_location_id  \
0  2025-08-13T15:57:55.015543+00:00             NaN          USGS-13185000   
1  2025-08-13T15:57:55.015543+00:00             NaN          USGS-13185000   
2  2025-08-13T15:57:55.015543+00:00             NaN          USGS-13185000   
3  2025-08-13T15:57:55.015543+00:00             NaN          USGS-13185000   
4  2025-08-13T15:57:55.015543+00:00             NaN          USGS-13185000   

   parameter_code qualifier  statistic_id                       time  \
0              60       NaN            11  1986-10-08T06:00:00+00:00   
1              60       NaN     

In [ ]:
import pandas as pd

df = pd.read_csv(
    "Twinsprings_alltime.csv",
    usecols=["time", "value"],
)

df["time"] = pd.to_datetime(df["time"], utc=True)
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = (
    df.dropna(subset=["time", "value"])
      .sort_values("time")
      .reset_index(drop=True)
)

# Calculate volume between each pair of consecutive 15-minute measurements.
interval = df["time"].shift(-1) - df["time"]
next_flow = df["value"].shift(-1)
valid = interval.eq(pd.Timedelta(minutes=15)) & next_flow.notna()

midpoint = df["time"] + interval / 2
volume_acre_ft = (
    (df["value"] + next_flow) / 2
    * interval.dt.total_seconds()
    / 43560
)

intervals = pd.DataFrame({
    "year": midpoint.dt.year,
    "volume_acre_ft": volume_acre_ft,
    "observed_hours": interval.dt.total_seconds() / 3600,
}).loc[valid]

annual = intervals.groupby("year").agg(
    annual_total_acre_ft=("volume_acre_ft", "sum"),
    observed_hours=("observed_hours", "sum"),
)

print(annual)
annual.to_csv("twinsprings_annual_streamflow.csv")

#The observed hours column helps show years with missing data. Lower numbers have less coverage. 

        annual_total_acre_ft  observed_hours
year                                        
1986.0          5.823807e+04         1624.00
1987.0          4.081874e+05         8004.00
1988.0          3.978699e+05         7030.00
1989.0          6.452461e+05         7404.25
1990.0          4.871035e+05         7047.50
1991.0          3.467912e+05         6113.50
1992.0          3.442734e+05         6673.25
1993.0          7.187624e+05         5003.50
1994.0          4.031160e+05         7824.00
1995.0          9.251945e+05         7078.25
1996.0          1.171834e+06         8106.50
1997.0          1.434849e+06         8089.25
1998.0          9.006612e+05         8062.25
1999.0          8.449914e+05         7629.75
2000.0          7.352217e+05         8101.00
2001.0          3.970733e+05         7776.00
2002.0          7.200104e+05         8470.00
2003.0          7.727149e+05         8524.75
2004.0          6.551065e+05         8059.50
2005.0          6.033551e+05         7985.00
2006.0    